In [1]:
import os
import glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import json
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm

# collect scores

In [ ]:
# Base directory containing all expression runs
base_dir = "../../runs/expression"
# base_dir = "/runs/expression" # for mb machine

# Function to extract scores from TensorBoard event files
def extract_scores_from_events(event_file_path, run_name):
	"""Extract all scalar values at a specific step from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			# Find events at or closest to target step
			step_values = [(event.step, event.value) for event in events]
						
			for step, value in step_values:
				if step > 100: # skip test runs
					scores.append({"step": step, "value": value, "run_name": run_name, "metric": tag})
		return scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return {}

# Main function to traverse all subdirectories and collect scores
def collect_all_scores(base_dir):
    """Traverse all subdirectories recursively and collect scores at specified step."""
    all_results = []
    all_dirs = list(os.walk(base_dir))
    for ind, (root, dirs, files) in enumerate(all_dirs):
        # Find TensorBoard event files in this directory
        event_files = glob.glob(os.path.join(root, "events.out.tfevents*"))
        if not event_files:
            continue
        print (ind, " of ",len(all_dirs))            
        # print(f"Processing: {root}")
        for event_file in event_files:
            # Only keep the part of the path after base_dir in run_name
            rel_run_name = os.path.relpath(root, base_dir)
            scores = extract_scores_from_events(event_file, run_name=rel_run_name)
            if scores:
                all_results.extend(scores)
                # print(f"  Found {len(scores)} metrics")
            else:
                pass
                # print(f"  No scores found")
    return all_results
	
# Execute the collection
print(f"Starting to collect scores from all subdirectories...")
results = collect_all_scores(base_dir)

print(f"\nCollected data from {len(results)} runs")

# Convert to DataFrame for easier analysis
if results:
	df = pd.DataFrame.from_records(results)
	print(f"\nDataFrame shape: {df.shape}")
	print("\nColumns:", df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(df.head())
	
	# Save to CSV
	output_file = f"expression_scores.csv"
	df.to_csv(output_file, index=False, compression="gzip")
	print(f"\nResults saved to {output_file}")
else:
	print("No results found!")

Starting to collect scores from all subdirectories...
2  of  2837
5  of  2837
7  of  2837
9  of  2837
10  of  2837
11  of  2837
12  of  2837
14  of  2837
15  of  2837
16  of  2837
17  of  2837
19  of  2837
20  of  2837
21  of  2837
22  of  2837
23  of  2837
28  of  2837
29  of  2837
31  of  2837
32  of  2837
33  of  2837
34  of  2837
35  of  2837
36  of  2837
37  of  2837
39  of  2837
40  of  2837
41  of  2837
44  of  2837
45  of  2837
48  of  2837
49  of  2837
54  of  2837
56  of  2837
57  of  2837
58  of  2837
61  of  2837
63  of  2837
64  of  2837
65  of  2837
67  of  2837
69  of  2837
70  of  2837
74  of  2837
719  of  2837
723  of  2837
2797  of  2837
2800  of  2837
2801  of  2837
2804  of  2837
2806  of  2837
2807  of  2837
2809  of  2837
2814  of  2837
2817  of  2837
2821  of  2837
2822  of  2837
2829  of  2837
2830  of  2837
2833  of  2837

Collected data from 92344 runs

DataFrame shape: (92344, 4)

Columns: ['step', 'value', 'run_name', 'metric']

First few results:


,step,value,run_name,metric
0,250,11.555820,20250413-202948,loss/iterations/train
1,500,11.434798,20250413-202948,loss/iterations/train
2,750,11.261512,20250413-202948,loss/iterations/train
3,1000,11.167288,20250413-202948,loss/iterations/train
4,1250,10.246181,20250413-202948,loss/iterations/train



Results saved to expression_scores.csv


In [4]:
df["run_name"].unique()

array(['20250413-202948', '20250419-222413',
       'upsample/ctloss/one_hot/20250715-175238',
       'upsample/ctloss/logemb/20250715-175744', '20250413-220701'],
      dtype=object)

# Output scores for selected run in table-ready format

In [ ]:
df = pd.read_csv("expression_scores.csv", compression="gzip")
# df = pd.read_csv("expression_scores_25-07-2025.csv", compression="gzip")
df.head(2)

,step,value,run_name,metric
0,250,11.555820,20250413-202948,loss/iterations/train
1,500,11.434798,20250413-202948,loss/iterations/train


In [21]:
df["run_name"].unique()

array(['20250413-202948', '20250419-222413',
       'upsample/ctloss/one_hot/20250715-175238',
       'upsample/ctloss/logemb/20250715-175744', '20250413-220701'],
      dtype=object)

In [26]:
selected_run = 'upsample/ctloss/one_hot/20250715-175238'

In [27]:
sel = df.query("run_name == @selected_run")
_met = [i for i in sel.metric.unique() if ("valid" in i) and ("v1" in i or "loss" in i) and ("samples" in i) and not ("_4_" in i)]
sel = sel.query("metric in @_met").dropna(subset=["value"])
results = sel.groupby("metric").apply(lambda x: x.loc[x["value"].idxmin()] if x["metric"].str.contains("loss").any() else x.loc[x["value"].idxmax()])
results.reset_index(drop=True).sort_values("metric", ascending=False)

,step,value,run_name,metric
9,1056000,0.380246,upsample/ctloss/one_hot/20250715-175238,score_predictions_Expression_dataset_v1_mm10_C...
8,1104000,0.092996,upsample/ctloss/one_hot/20250715-175238,score_predictions_Expression_dataset_v1_GRCh38...
7,128000,0.566972,upsample/ctloss/one_hot/20250715-175238,pearson_corr_genes_Expression_dataset_v1_mm10_...
6,192000,0.569982,upsample/ctloss/one_hot/20250715-175238,pearson_corr_genes_Expression_dataset_v1_GRCh3...
5,1104000,0.051568,upsample/ctloss/one_hot/20250715-175238,pearson_corr_cells_Expression_dataset_v1_mm10_...
4,4144000,0.064009,upsample/ctloss/one_hot/20250715-175238,pearson_corr_cells_Expression_dataset_v1_GRCh3...
3,160000,1.637746,upsample/ctloss/one_hot/20250715-175238,loss_mean_loss/samples/valid
2,320000,0.422170,upsample/ctloss/one_hot/20250715-175238,loss_diviation_loss/samples/valid
1,160000,0.482955,upsample/ctloss/one_hot/20250715-175238,loss_cls_loss/samples/valid
0,160000,0.480990,upsample/ctloss/one_hot/20250715-175238,loss/samples/valid


In [28]:
# select steps when scpre prediction on v1 is best (2 steps probably, one for human, one for mouse)
steps = results[results.metric.str.startswith('score_predictions_Expression_dataset_v1')]["step"].values

In [29]:
for step in steps:
	# sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)
	_met_sorted = [
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]
	print (selected_run, end="\t")
	print (step, end="\t")
	for met in _met_sorted:
		# print (met, end="\t")
		v = sel.query("metric == @met and step == @step")["value"].values[0]
		print (v, end="\t")
	print("\n------")

upsample/ctloss/one_hot/20250715-175238	1104000	0.0929964110255241	0.3637245893478393	-0.0057093477807939	0.0515678413212299	0.4368117153644562	0.4170084595680237	
------
upsample/ctloss/one_hot/20250715-175238	1056000	0.0564592368900775	0.3802457451820373	0.009273549541831	0.0491683222353458	0.4462359249591827	0.4339765906333923	
------


44800
0.1085683479905128	0.110475406050682	-0.0988152027130127	-0.0105760768055915	0.5893904566764832	0.5265839695930481	

In [62]:
_met

['loss/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid']